# WandB Run Comparison by ID

Fetches specific experiment runs from WandB and compares metrics, heatmaps, and ridge plots.

- **Metrics table** — key comparison metrics for all runs, grouped by dataset
- **Heatmap** — one combined heatmap per dataset with all CFs from every run, labeled by run name
- **Ridge plot** — one individual ridge plot per run, showing that run's CFs against the dataset distribution

In [ ]:
# Configuration — WandB run IDs to compare
RUN_IDS = ['bgyxnb4h', 'zkljo11n', 'srjd3c3b', 'lpkj67ca', 'yv2ab6ib', 'boqdxzpx', '7528cjjg', 'ggwlo4xd']

ENTITY = 'mllab-ts-universit-di-trieste'
PROJECT = 'CounterFactualDPG'


In [ ]:
import sys
import os
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

# Add counterfactual root to path
_cf_root = os.path.abspath('..')
_repo_root = os.path.abspath(os.path.join('..', '..'))
for p in [_cf_root, _repo_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

import wandb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from counterfactual_visualizer import (
    heatmap_techniques,
    plot_ridge_comparison,
    plot_pca_with_counterfactuals_clean,
)
from utils.compare_techniques import COMPARISON_METRICS
from utils.dataset_loader import load_dataset
from utils.config_manager import load_config

print('Imports OK')

## Fetch runs from WandB

In [ ]:
api = wandb.Api(timeout=60)

runs_info = {}  # run_id -> {meta, config, summary, metrics}

for run_id in RUN_IDS:
    run = api.run(f'{ENTITY}/{PROJECT}/{run_id}')
    config = run.config
    summary = run.summary._json_dict

    data_cfg = config.get('data', {})
    dataset_name = data_cfg.get('dataset_name') or data_cfg.get('dataset', 'unknown')

    # Infer technique
    technique = data_cfg.get('method', '')
    if not technique:
        name_lower = run.name.lower()
        if '_dpg' in name_lower or name_lower.endswith('dpg'):
            technique = 'dpg'
        elif '_dice' in name_lower or name_lower.endswith('dice'):
            technique = 'dice'

    # Extract comparison metrics
    metrics = {}
    for metric_key, metric_info in COMPARISON_METRICS.items():
        for wkey in metric_info.get('wandb_keys', [f'metrics/{metric_key}']):
            if wkey in summary:
                metrics[metric_key] = summary[wkey]
                break

    # Extract sample, counterfactuals, constraints
    feature_names = config.get('feature_names', [])

    sample = config.get('sample') or summary.get('sample') or summary.get('original_sample')
    if isinstance(sample, list) and feature_names:
        sample = dict(zip(feature_names, sample))

    sample_class = config.get('sample_class')
    if sample_class is None:
        sample_class = summary.get('sample_class') or summary.get('original_class')

    cfs = summary.get('final_counterfactuals', [])
    if isinstance(cfs, str):
        try:
            cfs = json.loads(cfs)
        except Exception:
            cfs = []
    if cfs and isinstance(cfs[0], list) and feature_names:
        cfs = [dict(zip(feature_names, cf)) for cf in cfs]

    restrictions = config.get('restrictions')
    constraints = config.get('dpg', {}).get('constraints') if config.get('dpg') else None

    runs_info[run_id] = {
        'name': run.name,
        'state': run.state,
        'dataset': dataset_name,
        'technique': technique.lower() if technique else 'unknown',
        'metrics': metrics,
        'sample': sample,
        'sample_class': sample_class,
        'counterfactuals': cfs,
        'restrictions': restrictions,
        'constraints': constraints,
        'feature_names': feature_names,
    }
    print(f'  {run_id}  {run.name}  dataset={dataset_name}  technique={technique}  '
          f'state={run.state}  #cfs={len(cfs)}  #metrics={len(metrics)}')

print(f'\nFetched {len(runs_info)} runs.')

## Group runs by dataset

In [ ]:
from collections import defaultdict

# Group runs by dataset (for combined heatmap)
dataset_runs = defaultdict(list)  # {dataset: [run_id, ...]}

for run_id, info in runs_info.items():
    dataset_runs[info['dataset']].append(run_id)

for ds, rids in dataset_runs.items():
    names = [f"{runs_info[r]['name']} ({runs_info[r]['technique']})" for r in rids]
    print(f'{ds}: {len(rids)} runs — {", ".join(names)}')

## Metrics comparison table

In [ ]:
KEY_METRICS = [
    'perc_valid_cf_all', 'perc_actionable_cf_all',
    'plausibility_nbr_cf', 'distance_mh',
    'avg_nbr_changes', 'count_diversity_all',
    'accuracy_knn_sklearn', 'runtime',
]

# Build column labels with ↑/↓ direction indicators from COMPARISON_METRICS goals
col_labels = {}   # metric_key -> display label
col_goals  = {}   # display label -> goal
for k in KEY_METRICS:
    info = COMPARISON_METRICS.get(k, {})
    arrow = '↑' if info.get('goal') == 'maximize' else '↓'
    label = f"{info.get('name', k)} {arrow}"
    col_labels[k] = label
    col_goals[label] = info.get('goal', 'maximize')

def bold_best(s):
    goal = col_goals.get(s.name, 'maximize')
    numeric = pd.to_numeric(s, errors='coerce')
    if numeric.isna().all():
        return ['' for _ in s]
    best = numeric.max() if goal == 'maximize' else numeric.min()
    return ['font-weight: bold' if (pd.notna(v) and v == best) else '' for v in numeric]

for ds, rids in sorted(dataset_runs.items()):
    display(Markdown(f'### {ds}'))
    rows = []
    for rid in rids:
        info = runs_info[rid]
        row = {'run': info['name']}
        for k in KEY_METRICS:
            val = info['metrics'].get(k)
            row[col_labels[k]] = round(val, 4) if val is not None else None
        rows.append(row)
    if rows:
        df_table = pd.DataFrame(rows).set_index('run')
        display(df_table.style.apply(bold_best, axis=0).format('{:.4f}', na_rep='-'))



print('proximity - diversity')

## Heatmap — all runs per dataset

One combined heatmap per dataset showing all counterfactuals from every run, labeled by run name.

In [ ]:
import seaborn as sns

for ds, rids in sorted(dataset_runs.items()):
    # Use the first run's sample as the reference
    ref = runs_info[rids[0]]
    sample = ref['sample']
    sample_class = ref['sample_class']
    restrictions = ref['restrictions']

    if not sample:
        print(f'⚠ {ds}: no sample found, skipping')
        continue

    display(Markdown(f'---\n## {ds}'))

    # Build combined DataFrame: Original + CFs from each run
    sample_df = pd.DataFrame([sample], index=['Original'])
    all_rows = [sample_df]

    for rid in rids:
        info = runs_info[rid]
        run_label = info['name']
        for i, cf in enumerate(info['counterfactuals'][:5]):
            cf_df = pd.DataFrame([cf], index=[f'{run_label} CF {i+1}'])
            all_rows.append(cf_df)

    full_df = pd.concat(all_rows)
    full_df = full_df[sorted(full_df.columns)]

    # Difference matrix for color mapping
    diff_matrix = full_df.copy()
    diff_matrix.iloc[0] = 0
    for i in range(1, len(full_df)):
        diff_matrix.iloc[i] = full_df.iloc[i] - full_df.iloc[0]

    vmax = np.max(np.abs(diff_matrix.iloc[1:].values)) if len(diff_matrix) > 1 else 1
    if vmax == 0:
        vmax = 1

    fig = plt.figure(figsize=(14, max(6, 1.5 + 0.45 * len(full_df))))
    ax = sns.heatmap(
        diff_matrix,
        annot=full_df.values,
        fmt='.2f',
        cmap='coolwarm',
        cbar=True,
        linewidths=1.0,
        linecolor='k',
        vmin=-vmax,
        vmax=vmax,
        cbar_kws={'label': 'Magnitude difference'},
    )

    if restrictions:
        ordered_restrictions = {feat: restrictions[feat] for feat in sorted(full_df.columns) if feat in restrictions}
        symbol_map = {'no_change': '⊝', 'non_increasing': '⬇️', 'non_decreasing': '⬆️'}
        for i, (feat, restr) in enumerate(ordered_restrictions.items()):
            if restr in symbol_map:
                ax.text(i + 0.5, len(full_df) + 0.5, symbol_map[restr],
                        ha='center', va='top', fontweight='bold', fontsize=14)

    plt.title(f'{ds} — All Runs (Original Class {sample_class})', fontsize=14, fontweight='bold')
    plt.xticks(rotation=35, ha='right')
    plt.yticks(rotation=0, va='center')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## Pairwise heatmap comparisons

Each pair compares counterfactuals from two specific runs side by side using `heatmap_techniques`.

In [ ]:
# Build a lookup: run_name -> run_id
name_to_id = {info['name']: rid for rid, info in runs_info.items()}
print('Available run names:')
for name, rid in name_to_id.items():
    print(f'  {name} -> {rid}')

# Define pairwise comparisons (run_name_A, run_name_B)
HEATMAP_PAIRS = [
    ('iris_dpg', 'iris_dice'),
    ('iris_dpg', 'iris_dpg_dice'),
    ('iris_dice', 'iris_dpg_dice'),
    ('iris_dpg', 'iris_dpg_new_constraints'),
    ('iris_dpg_dice', 'iris_dpg_dice_new_constraints'),
]

for name_a, name_b in HEATMAP_PAIRS:
    rid_a = name_to_id.get(name_a)
    rid_b = name_to_id.get(name_b)

    if not rid_a or not rid_b:
        missing = [n for n, r in [(name_a, rid_a), (name_b, rid_b)] if not r]
        print(f'⚠ Skipping {name_a} x {name_b}: run(s) not found: {missing}')
        continue

    info_a = runs_info[rid_a]
    info_b = runs_info[rid_b]

    sample = info_a['sample']
    sample_class = info_a['sample_class']
    cfs_a = info_a['counterfactuals']
    cfs_b = info_b['counterfactuals']
    restrictions = info_a['restrictions']

    if not sample or not cfs_a or not cfs_b:
        print(f'⚠ Skipping {name_a} x {name_b}: missing sample or counterfactuals')
        continue

    display(Markdown(f'---\n### {name_a}  vs  {name_b}'))

    fig = heatmap_techniques(
        sample=sample,
        class_sample=sample_class,
        cf_list_1=cfs_a[:5],
        cf_list_2=cfs_b[:5],
        technique_names=(name_a, name_b),
        restrictions=restrictions,
    )
    if fig:
        display(fig)
        plt.close(fig)

## PCA & Ridge plots — one per run

Individual visualizations for each run:
- **PCA** — 2D projection of the dataset with the run's counterfactuals
- **Ridge** — feature distribution comparison against the dataset

In [ ]:
# Helper: load dataset and train a model (for ridge plot)
_dataset_cache = {}

def get_dataset_model(dataset_name):
    """Load dataset, train RF model, return dict with dataset_df, target, model."""
    if dataset_name in _dataset_cache:
        return _dataset_cache[dataset_name]

    config_path = os.path.join(_cf_root, 'configs', dataset_name, 'config.yaml')
    if not os.path.exists(config_path):
        print(f'  Config not found: {config_path}')
        return None

    config = load_config(config_path, repo_root=_cf_root)
    seed = getattr(config.experiment_params, 'seed', 42)
    np.random.seed(seed)

    ds = load_dataset(config, repo_root=_cf_root)
    features_df = ds['features_df']
    labels = ds['labels']

    X_train, X_test, y_train, y_test = train_test_split(
        features_df, labels, test_size=0.2, random_state=seed, stratify=labels
    )

    model = RandomForestClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train, y_train)

    result = {'dataset_df': features_df, 'target': labels, 'model': model}
    _dataset_cache[dataset_name] = result
    return result

print('Helper ready')

In [ ]:
for run_id, info in runs_info.items():
    ds = info['dataset']
    run_name = info['name']
    cfs = info['counterfactuals']
    sample = info['sample']
    constraints = info['constraints']

    if not sample or not cfs:
        print(f'⚠ {run_name}: missing sample or counterfactuals, skipping')
        continue

    dm = get_dataset_model(ds)
    if dm is None:
        print(f'  ⚠ {run_name}: could not load dataset/model')
        continue

    dataset_df = dm['dataset_df']
    target = dm['target']
    model = dm['model']

    cf_df = pd.DataFrame(cfs)
    sample_df = pd.DataFrame([sample])
    original_class = model.predict(sample_df)[0]
    cf_predicted_classes = model.predict(cf_df)
    target_class = cf_predicted_classes[0] if len(cf_predicted_classes) > 0 else None

    display(Markdown(f'---\n### {run_name}\n`{run_id}` — {info["technique"].upper()} on {ds}'))

    # --- PCA ---
    display(Markdown('#### PCA'))
    fig_pca = plot_pca_with_counterfactuals_clean(
        model=model,
        dataset=dataset_df,
        target=target,
        sample=sample,
        counterfactuals_df=cf_df,
        cf_predicted_classes=cf_predicted_classes,
    )
    if fig_pca:
        display(fig_pca)
        plt.close(fig_pca)

    # --- Ridge ---
    display(Markdown('#### Ridge'))
    fig_ridge = plot_ridge_comparison(
        sample=sample,
        cf_list_1=cfs,
        cf_list_2=[],
        technique_names=(run_name, ''),
        method_1_color='#CC0000',
        method_2_color='#006DAC',
        dataset_df=dataset_df,
        constraints=constraints,
        target=target,
        target_class=target_class,
        original_class=original_class,
        show_per_class_distribution=True,
        show_overall_distribution=False,
        show_original_class_constraints=False,
    )
    if fig_ridge:
        display(fig_ridge)
        plt.close(fig_ridge)